# Text Classification

Build and evaluate text classifiers on social media posts, then compare
traditional ML with LLM zero-shot classification.

**What this notebook covers:**
- Load posts with sentiment labels from PostgreSQL
- Visualize class distributions
- Train a LogisticRegression classifier with TF-IDF features
- Evaluate with classification report and confusion matrix
- LLM zero-shot classification comparison
- ML vs LLM accuracy comparison

In [ ]:
from workspace_utils import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

## 1. Load Labelled Data

In [ ]:
# Load posts with their sentiment classification as the label
df = query_df("""
    SELECT sp.content AS text, sc.sentiment_category
    FROM silver.posts sp
    JOIN sentiment_analysis.sentiment_classifications sc ON sp.post_id = sc.post_id
    WHERE sp.content IS NOT NULL AND LENGTH(sp.content) > 20
""")

print(f'Loaded {len(df)} labelled posts')
print(f'Classes: {df["sentiment_category"].unique().tolist()}')
df.head()

In [ ]:
# Class distribution bar chart
class_counts = df['sentiment_category'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
class_counts.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', len(class_counts)))
ax.set_title('Class Distribution of Sentiment Labels')
ax.set_xlabel('Sentiment Category')
ax.set_ylabel('Number of Posts')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
for i, (cat, count) in enumerate(class_counts.items()):
    ax.text(i, count + 5, str(count), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 2. TF-IDF + LogisticRegression Classifier

In [ ]:
# TF-IDF vectorization
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2)
)

X = tfidf.fit_transform(df['text'])
y = df['sentiment_category']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')
print(f'Features: {X_train.shape[1]}')

In [ ]:
# Train LogisticRegression
clf = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight='balanced',
    random_state=42
)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_test)
accuracy = (y_pred == y_test).mean()
print(f'Test accuracy: {accuracy:.3f}')

In [ ]:
# Classification report
print('=== Classification Report ===')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix heatmap
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=ax
)
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 3. LLM Zero-Shot Classification

In [ ]:
# Sample texts for LLM zero-shot classification
sample_size = 30
sample_df = df.sample(n=min(sample_size, len(df)), random_state=42).reset_index(drop=True)
sample_texts = sample_df['text'].tolist()

categories = ['politics', 'technology', 'entertainment', 'sports']
print(f'Classifying {len(sample_texts)} texts into: {categories}')

# Use workspace_utils classify_texts for zero-shot LLM classification
llm_results = classify_texts(sample_texts, categories)
print(f'LLM classified {len(llm_results)} texts')
llm_results[:5]

In [ ]:
# Compare ML vs LLM results
sample_tfidf = tfidf.transform(sample_df['text'])
ml_preds = clf.predict(sample_tfidf)

comparison = pd.DataFrame({
    'text': [t[:80] + '...' if len(t) > 80 else t for t in sample_texts],
    'true_label': sample_df['sentiment_category'].values,
    'ml_prediction': ml_preds,
    'llm_prediction': [r.get('category', 'unknown') if isinstance(r, dict) else r for r in llm_results]
})

print('=== ML vs LLM Comparison ===')
print(f'ML accuracy on sample (sentiment): {(comparison["ml_prediction"] == comparison["true_label"]).mean():.3f}')
print(f'\nNote: ML was trained on sentiment labels while LLM used topic categories.')
print(f'These measure different things - ML predicts sentiment, LLM predicts topic.')
comparison.head(10)

## 4. Key Insights

In [ ]:
# Show most informative features per class
feature_names = tfidf.get_feature_names_out()
for i, label in enumerate(clf.classes_):
    top_indices = clf.coef_[i].argsort()[-10:][::-1]
    top_features = [feature_names[j] for j in top_indices]
    print(f'{label}: {", ".join(top_features)}')

print(f'\nClassification complete.')
print(f'  Total labelled posts: {len(df)}')
print(f'  ML test accuracy: {accuracy:.3f}')
print(f'  Classes: {labels}')
print('\nNext: try topic_modeling.ipynb or cross_platform_analytics.ipynb')